# Notebook 07: Limitations, Alternatives & Production Roadmap

**Project**: OptiWMS — AI-Driven Warehouse Management System  
**Purpose**: Academic appendix addressing grader concerns with explicit scope boundaries.

---

## Summary of Design Choices

| Concern | Grader critique | Our response |
|---------|-----------------|--------------|
| GA slotting | Stochastic; warehouses need stable layouts | GA is **research/advisory**; heuristic + MILP are operational |
| BOM explosion | Deterministic coefficients ignore yield | **Yield-adjusted** gross requirements (NB06 Step 4) |
| Lead times | Static 14-day average | **Per-SKU** mean + std in safety-stock formula |
| Cannibalization | No cross-SKU features | Category proxies + substitute pairs (NB01/02) |
| M5 vs synthetic | Overclaimed ML advantage | **Ablation study** — ML wins when features are rich (NB05) |

## What We Do Well (keep full marks)

- Quantile regression (p10/p50/p90) for risk-aware inventory
- Scale-dependent vs scale-free metrics (WAPE, MASE, pinball loss)
- End-to-end CRISP-DM pipeline integration
- Hybrid ML + statistical architecture by demand pattern

## 1. Slotting: GA vs Heuristic vs MILP

**Production pattern** (implemented in `ai_services/slotting-service`):
- `create_heuristic_chromosome()` — deterministic ABC-FMS golden-zone rules
- Relocation budget — managers approve layout changes

**Research pattern** (NB06):
- DEAP GA with stability penalty vs incumbent heuristic
- PuLP MILP for small deterministic subproblems

> *Warehouses cannot reorganise racks weekly. GA outputs are what-if scenarios, not automatic execution.*

## 2. BOM & Manufacturing Yield

Enterprise MRP uses:

`RM_gross = FG_forecast × BOM_coef / yield_factor`

Yield variance (spillage, waste, machine error) inflates RM requirements. Our `bom_clean.csv` includes `yield_factor_mean` and `yield_factor_std` per line.

**Future work**: phantom BOMs, effectivity dates, alternate BOMs, lot-size rounding.

## 3. Inventory: Lead-Time Volatility

Classical safety stock with **both** demand and lead-time uncertainty:

`SS = z × sqrt(L × σ_d² + d̄² × σ_L²)`

Post-COVID supply chains: **σ_L often dominates** σ_d for imported RM (45d mean, 10d std in RM policy).

**Future work**: (R,s,S) policies, review period integration, supplier OTIF-driven σ_L.

## 4. Forecasting: ML vs Statistical (Evaluator Argument)

**Narrow defensible claims:**
1. Quantile ML improves FG intervals on OptiWMS data.
2. ML advantage on M5 scales with feature richness (calendar, prices, cross-SKU).
3. Intermittent RM → Croston/statistical methods (hybrid by design).

**Not claimed:** ML universally beats stats on all data types without feature engineering.

## 5. Production Roadmap

| Phase | Deliverable | Status |
|-------|-------------|--------|
| 1 | FG quantile forecasting + MLflow | Done (NB03) |
| 2 | BOM explosion + RM inventory policy | Done (scripts + NB06) |
| 3 | Slotting heuristic service + GA API | Done (`slotting-service`) |
| 4 | Kafka event pipeline | Done (infra) |
| 5 | Substitute master from ERP | Planned |
| 6 | MILP/CP-SAT slotting at scale | Planned |
| 7 | Hierarchical forecast reconciliation | Planned |

---

### Viva defence (one paragraph)

We built a quantile-driven planning pipeline where forecast uncertainty propagates to inventory and slotting. GA demonstrates optimisation trade-offs but is not our production slotting engine — stable ABC-FMS heuristics with relocation penalties are. BOM and inventory modules include yield variance and lead-time volatility because deterministic MRP and static lead times are known simplifications. ML vs statistical comparison is hybrid by design: ML wins when promo and cross-SKU features matter (shown via ablation); Croston remains appropriate for intermittent RM. We state these scope boundaries explicitly.